In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
import joblib
import os

In [4]:
df = pd.read_csv('../Data/group5-adult.csv')

# Clean Missing Values

In [5]:
df.replace('?', np.nan, inplace=True)
print("Missing values after cleaning:")
print(df.isnull().sum())

Missing values after cleaning:
age               0
workclass         0
fnlwgt            0
education         0
education-num     0
marital-status    0
occupation        0
relationship      0
race              0
sex               0
capital-gain      0
capital-loss      0
hours-per-week    0
native-country    0
income            0
dtype: int64


# Encode Target Variable

In [6]:
df['income'] = df['income'].str.strip()
df['income'] = (df['income'] == '>50K').astype(int)

print("Target value counts:")
print(df['income'].value_counts())
print(f"\nClass 0 (<=50K): {(df['income']==0).sum()}")
print(f"Class 1 (>50K) : {(df['income']==1).sum()}")

Target value counts:
income
0    22328
1     7057
Name: count, dtype: int64

Class 0 (<=50K): 22328
Class 1 (>50K) : 7057


# Define Feature Columns

In [7]:
numeric_cols = [
    'age', 'fnlwgt', 'education-num',
    'capital-gain', 'capital-loss', 'hours-per-week'
]

categorical_cols = [
    'workclass', 'education', 'marital-status',
    'occupation', 'relationship', 'race',
    'sex', 'native-country'
]

X = df[numeric_cols + categorical_cols]
y = df['income']

print(f"Feature matrix shape : {X.shape}")
print(f"Target shape         : {y.shape}")
print(f"Numeric features     : {numeric_cols}")
print(f"Categorical features : {categorical_cols}")

Feature matrix shape : (29385, 14)
Target shape         : (29385,)
Numeric features     : ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
Categorical features : ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country']


 # Train/Test Split

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X_train shape : {X_train.shape}")
print(f"X_test shape  : {X_test.shape}")
print(f"y_train distribution:\n{y_train.value_counts()}")
print(f"\ny_test distribution:\n{y_test.value_counts()}")

X_train shape : (23508, 14)
X_test shape  : (5877, 14)
y_train distribution:
income
0    17862
1     5646
Name: count, dtype: int64

y_test distribution:
income
0    4466
1    1411
Name: count, dtype: int64


# Build Preprocessing Pipeline

In [9]:
# Numeric pipeline: impute median + scale
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical pipeline: impute most frequent + one-hot encode
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine both
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, categorical_cols)
])

print("Preprocessing pipeline built!")
print(preprocessor)

Preprocessing pipeline built!
ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['age', 'fnlwgt', 'education-num',
                                  'capital-gain', 'capital-loss',
                                  'hours-per-week']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['workclass', 'education', 'marital-status',
      

# Fit & Verify Pipeline

In [10]:
# Fit on training data only
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

print(f"X_train transformed shape : {X_train_transformed.shape}")
print(f"X_test transformed shape  : {X_test_transformed.shape}")
print(" Pipeline fitted successfully!")

X_train transformed shape : (23508, 107)
X_test transformed shape  : (5877, 107)
 Pipeline fitted successfully!


In [11]:
# Save train/test splits
joblib.dump((X_train, X_test, y_train, y_test), 'data/train_test_split.pkl')

# Save fitted preprocessor
joblib.dump(preprocessor, 'data/preprocessor.pkl')

# Save column lists
joblib.dump(numeric_cols, 'data/numeric_cols.pkl')
joblib.dump(categorical_cols, 'data/categorical_cols.pkl')

print("All shared files saved to notebooks/data/:")
print("   - data/train_test_split.pkl")
print("   - data/preprocessor.pkl")
print("   - data/numeric_cols.pkl")
print("   - data/categorical_cols.pkl")

All shared files saved to notebooks/data/:
   - data/train_test_split.pkl
   - data/preprocessor.pkl
   - data/numeric_cols.pkl
   - data/categorical_cols.pkl


In [12]:
# Test reload
X_train_r, X_test_r, y_train_r, y_test_r = joblib.load('data/train_test_split.pkl')
preprocessor_r = joblib.load('data/preprocessor.pkl')
numeric_cols_r = joblib.load('data/numeric_cols.pkl')
categorical_cols_r = joblib.load('data/categorical_cols.pkl')

print("All files reloaded successfully!")
print(f"X_train shape : {X_train_r.shape}")
print(f"X_test shape  : {X_test_r.shape}")
print(f"Numeric cols  : {numeric_cols_r}")
print(f"Cat cols      : {categorical_cols_r}")

All files reloaded successfully!
X_train shape : (23508, 14)
X_test shape  : (5877, 14)
Numeric cols  : ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
Cat cols      : ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country']


In [13]:
print("=" * 55)
print("PREPROCESSING PIPELINE SUMMARY")
print("=" * 55)
print(f"Total samples        : {len(X)}")
print(f"Training samples     : {len(X_train)}")
print(f"Test samples         : {len(X_test)}")
print(f"Numeric features     : {len(numeric_cols)}")
print(f"Categorical features : {len(categorical_cols)}")
print(f"Transformed shape    : {X_train_transformed.shape[1]} features after encoding")
print("=" * 55)
print("Ready! All members can now load from data/ folder.")

PREPROCESSING PIPELINE SUMMARY
Total samples        : 29385
Training samples     : 23508
Test samples         : 5877
Numeric features     : 6
Categorical features : 8
Transformed shape    : 107 features after encoding
Ready! All members can now load from data/ folder.
